# logsumexp-cross-entropy composite — cx17: full logsumexp by hand — max-shift + sum + log + broadcast-add-back

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `logsumexp-cross-entropy`, `sum-and-broadcast-duality`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "logsumexp-cross-entropy"
DD_ATOM_IDS = ["logsumexp-cross-entropy", "sum-and-broadcast-duality"]
DD_SUBTOPICS = ["Loss: logsumexp cross-entropy", "Backprop: sum/broadcast duality"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Hand-rolled `logsumexp` — sum/log/broadcast all wired together

1. **`sum-and-broadcast-duality`** — the keepdim flag controls whether the
   reduced axis is dropped or kept as size 1. Keep it when you need to
   broadcast the reduction result back over the original axis (subtract max
   from every class logit; add max back to log-sum-exp).
2. **`logsumexp-cross-entropy`** — the full identity is
   `log(sum(exp(x - m))) + m` with `m = max(x)`. The shift keeps every exp
   argument ≤ 0; the `+ m` restores the absolute scale.

Composition: this is the full LSE, including BOTH the keepdim-for-subtract
step AND the keepdim-for-add-back step. cx15 was the per-row scalar version;
this one returns a kept-dim shape so it composes downstream (e.g. softmax).


### Composite Exercise — full logsumexp by hand — max-shift + sum + log + broadcast-add-back

**Atoms exercised together**: `logsumexp-cross-entropy`, `sum-and-broadcast-duality`

Implement `cx17_logsumexp(x, dim, keepdim=False)` — the full numerically
stable `logsumexp` reducer along an arbitrary `dim`.

Algorithm:
1. `m = x.max(dim=dim, keepdim=True).values` (always kept for the subtract).
2. `shifted = x - m` (broadcast subtract over `dim`).
3. `s = shifted.exp().sum(dim=dim, keepdim=True)` (kept for the add-back).
4. `lse = s.log() + m` (kept-dim form).
5. If `keepdim=False`, squeeze `dim`.

Constraints:
- Do NOT call `torch.logsumexp` (this drill is about writing it).
- Must handle large-magnitude inputs (~10000) without overflow.
- Must honour the `keepdim` flag.


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx17_logsumexp(x: Tensor, dim: int, keepdim: bool = False) -> Tensor:
    """Hand-rolled logsumexp along `dim`, with full keepdim handling."""
    raise NotImplementedError()


def _test_cx17():
    import inspect
    import math
    src = inspect.getsource(cx17_logsumexp)
    assert 't.logsumexp' not in src and 'torch.logsumexp' not in src, (
        'must not delegate to torch.logsumexp'
    )

    rng = t.Generator().manual_seed(0)

    # --- dim=-1, keepdim=False ---
    X = t.randn(5, 8, generator=rng) * 3.0
    ours = cx17_logsumexp(X, dim=-1, keepdim=False)
    ref = t.logsumexp(X, dim=-1, keepdim=False)
    assert ours.shape == ref.shape == (5,)
    assert t.allclose(ours, ref, atol=1e-5), f'dim=-1 noKeep: {ours} vs {ref}'

    # --- dim=-1, keepdim=True ---
    ours_k = cx17_logsumexp(X, dim=-1, keepdim=True)
    ref_k = t.logsumexp(X, dim=-1, keepdim=True)
    assert ours_k.shape == ref_k.shape == (5, 1)
    assert t.allclose(ours_k, ref_k, atol=1e-5)

    # --- dim=0, keepdim=False ---
    X3 = t.randn(4, 3, 5, generator=rng) * 2.0
    ours0 = cx17_logsumexp(X3, dim=0, keepdim=False)
    ref0 = t.logsumexp(X3, dim=0, keepdim=False)
    assert ours0.shape == ref0.shape == (3, 5)
    assert t.allclose(ours0, ref0, atol=1e-5)

    # --- dim=1, keepdim=True ---
    ours1 = cx17_logsumexp(X3, dim=1, keepdim=True)
    ref1 = t.logsumexp(X3, dim=1, keepdim=True)
    assert ours1.shape == ref1.shape == (4, 1, 5)
    assert t.allclose(ours1, ref1, atol=1e-5)

    # --- huge-magnitude safety (overflow check) ---
    big = t.tensor([[10000.0, 9999.0, 10001.0],
                    [-10000.0, -9999.0, -10001.0]])
    ours_big = cx17_logsumexp(big, dim=-1)
    assert t.isfinite(ours_big).all(), f'overflow! {ours_big}'
    ref_big = t.logsumexp(big, dim=-1)
    assert t.allclose(ours_big, ref_big, atol=1e-3)

    # --- uniform identity: logsumexp([0]*C) = log(C) ---
    u = t.zeros(2, 5)
    got_u = cx17_logsumexp(u, dim=-1)
    assert t.allclose(got_u, t.full((2,), math.log(5)), atol=1e-6)

    _dd_passed.add('cx17')

_test_cx17()

<details><summary>Show solution — cx17</summary>

```python
def cx17_logsumexp(x: Tensor, dim: int, keepdim: bool = False) -> Tensor:
    # atom: sum-and-broadcast-duality — keepdim controls the broadcast.
    m = x.max(dim=dim, keepdim=True).values   # kept for broadcast subtract
    shifted = x - m                            # broadcast subtract over dim
    s = shifted.exp().sum(dim=dim, keepdim=True)  # kept for broadcast add-back
    # atom: logsumexp-cross-entropy — log(sum(exp(shift))) + m
    out = s.log() + m
    if not keepdim:
        out = out.squeeze(dim)
    return out

```

Two keepdim broadcasts happen here: once to subtract `m` over `dim`, and
once to add it back after the log. That dual use of keepdim IS the
sum-and-broadcast-duality atom. The final squeeze just respects the user's
`keepdim` flag — the math always works in kept-dim form internally.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx17'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx17',
        'subtopics': ["Loss: logsumexp cross-entropy", "Backprop: sum/broadcast duality"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()